In [3]:
from sys import path as syspath
from os import path as ospath

syspath.insert(1, r'D:\ARTM\topic-modelling-attention\src')
syspath

['C:\\Users\\kn\\AppData\\Roaming\\uv\\python\\cpython-3.13.9-windows-x86_64-none\\python313.zip',
 'D:\\ARTM\\topic-modelling-attention\\src',
 'D:\\ARTM\\topic-modelling-attention\\src',
 'C:\\Users\\kn\\AppData\\Roaming\\uv\\python\\cpython-3.13.9-windows-x86_64-none\\DLLs',
 'C:\\Users\\kn\\AppData\\Roaming\\uv\\python\\cpython-3.13.9-windows-x86_64-none\\Lib',
 'C:\\Users\\kn\\AppData\\Roaming\\uv\\python\\cpython-3.13.9-windows-x86_64-none',
 'C:\\Users\\kn\\AppData\\Local\\uv\\cache\\builds-v0\\.tmpGHKOtm',
 '',
 'C:\\Users\\kn\\AppData\\Local\\uv\\cache\\builds-v0\\.tmpGHKOtm\\Lib\\site-packages',
 'C:\\Users\\kn\\AppData\\Local\\uv\\cache\\archive-v0\\ZXpirby7L7XYy-rk4EZTc\\Lib\\site-packages',
 'C:\\Users\\kn\\AppData\\Local\\uv\\cache\\archive-v0\\ZXpirby7L7XYy-rk4EZTc\\Lib\\site-packages\\win32',
 'C:\\Users\\kn\\AppData\\Local\\uv\\cache\\archive-v0\\ZXpirby7L7XYy-rk4EZTc\\Lib\\site-packages\\win32\\lib',
 'C:\\Users\\kn\\AppData\\Local\\uv\\cache\\archive-v0\\ZXpirby7L7XY

In [11]:
import jax
import jax.numpy as jnp
import numpy as np 

from sklearn.datasets import fetch_20newsgroups

import matplotlib.pyplot as plt
import seaborn as sns

from cartm.model import ContextTopicModel
from cartm.preprocessing import DatasetPreprocessor

This example will guide you through the basic interaction with the model.

## Data preprocessing

First, let's download a sample dataset and preprocess it with `DatasetPreprocessor`.

In [5]:
with open('./data/test_data.txt') as f:
    data = f.readlines()
print(f'Total number of documents in corpus: {len(data)}')
print(f'Total number of words in corpus: {sum([len(doc.split(" ")) for doc in data])}')

Total number of documents in corpus: 4
Total number of words in corpus: 34


In [6]:
preprocessor = DatasetPreprocessor()
tokenized_data, document_bounds = preprocessor.fit_transform(data)
print(f'Total number of document boundaries in preprocessed corpus: {len(document_bounds)}')
print(f'Total number of tokenized words in preprocessed corpus: {len(tokenized_data)}')

Total number of document boundaries in preprocessed corpus: 5
Total number of tokenized words in preprocessed corpus: 20


In [7]:
vocabulary = preprocessor.vocabulary
print(document_bounds)
#sum(len(doc) for doc in tokenized_data)
print(tokenized_data)
print(vocabulary)

[ 0  5  9 16 20]
[18 17 12  8 16  1 15  8  0 11 14  5  3  6  2 10 13  7  4  9]
{'begin': 0, 'best': 1, 'big': 2, 'day': 3, 'disconnect': 4, 'everi': 5, 'lead': 6, 'need': 7, 'new': 8, 'reconnect': 9, 'result': 10, 'small': 11, 'someth': 12, 'sometim': 13, 'step': 14, 'time': 15, 'today': 16, 'tri': 17, 'would': 18}


In [8]:
def _norm(x):
    _eps = 1e-12
    x = np.maximum(x, np.zeros_like(x))
    norm = x.sum(axis=0)
    x = np.where(norm > _eps, x / norm, np.zeros_like(x))
    return x

In [16]:
ctx_len = 3
attn_bounds = document_bounds
print(f"{attn_bounds=}")
vocab_size = len(vocabulary)
print(f"{vocab_size=}")
n_topics = 10

attn_bounds=Array([ 0,  5,  9, 16, 20], dtype=int32)
vocab_size=19


In [17]:
phi = jax.random.uniform(  # инициализация равномерным распределением матрицы Фи размерность (W,T)
            key=jax.random.key(42),
            shape=(vocab_size, n_topics),
        )
phi = _norm(phi)
print(f"{phi.shape=}")

phi.shape=(19, 10)


In [19]:
n_t = jnp.full(            # вектор-псевдоматрица размерностью (T, ) для хранения частотности топиков
            shape=(n_topics, ),    # инициализация равномерным
            fill_value=len(tokenized_data) / n_topics,
        )  # (T, )
print(f"{n_t=}")

n_t=Array([2., 2., 2., 2., 2., 2., 2., 2., 2., 2.], dtype=float32, weak_type=True)


In [43]:
# phi умножается element-wise на n_t (частотность токенов), нормализованная и транспонированная
# (T, W) * (T, 1) = (T, W) -> t -> (W, T)
print(f"{phi.T.shape=}")
print(f"{n_t[:, None].shape=}")
phi_hatch = _norm(phi.T * n_t[:, None]).T # (W, T)
print(f"{phi_hatch.shape=}")
# 19 строк для каждого токена с вероятностью соответствующего топика в 10 колонках
# токен 8 - "new"
print(f"{phi_hatch[8]=}")

phi.T.shape=(10, 19)
n_t[:, None].shape=(10, 1)
phi_hatch.shape=(19, 10)
phi_hatch[8]=array([0.00623954, 0.00542419, 0.04397687, 0.14882576, 0.19154426,
       0.0731359 , 0.13844424, 0.06082839, 0.10824747, 0.22333333],
      dtype=float32)


In [44]:
batch = tokenized_data
print(f"{batch=}")
phi_it_hatch = jnp.take_along_axis( 
            phi_hatch,
            indices=batch[:, None],         # получаем только строки
            axis=0,
        )
print(f"{phi_it_hatch.shape=}")
# токен 8 - "new" стоит в 3 и 7 позиции в батче (tokenized_data)
print(f"{phi_it_hatch[3]=}")
print(f"{phi_it_hatch[7]=}")
# phi_it_hatch - матрица строки для каждого токена по порядку в батче (tokenized_data) с 10 колонками вероятности топиков

batch=Array([18, 17, 12,  8, 16,  1, 15,  8,  0, 11, 14,  5,  3,  6,  2, 10, 13,
        7,  4,  9], dtype=int32)
phi_it_hatch.shape=(20, 10)
phi_it_hatch[3]=Array([0.00623954, 0.00542419, 0.04397687, 0.14882576, 0.19154426,
       0.0731359 , 0.13844424, 0.06082839, 0.10824747, 0.22333333],      dtype=float32)
phi_it_hatch[7]=Array([0.00623954, 0.00542419, 0.04397687, 0.14882576, 0.19154426,
       0.0731359 , 0.13844424, 0.06082839, 0.10824747, 0.22333333],      dtype=float32)


In [45]:
def _get_context_tensor(batch):
    batch_size = batch.shape[0]     # размерность батча - количество токенов
    pad_token = -1  # assuming we don't have negative tokens in vocabulary
    
    # shifts for rolling the batch along new dimension
    #  [0, -1, -2, ..., -2 * ctx_len - 1]
    shifts = jnp.arange(0, -2 * ctx_len - 1, -1)  # (2C + 1, )
    print(f"{shifts=}")
    
    padding = jnp.full(
        (ctx_len, n_topics),
        fill_value=pad_token,
        dtype=batch.dtype,
    )  # (C, T)
    print(f"{padding.shape=}")
    print(f"{padding=}")
    
    padded_batch = jnp.concatenate([padding, batch, padding], axis=0)
    print(f"{padded_batch.shape=}")
    print(f"{padded_batch=}")
    
    # rolling and clipping each "slice" of batch
    def shift_batch(shift):
        return jnp.roll(padded_batch, shift, axis=0)[:batch_size]
    
    # apply vmap over all shifts
    # размножение батча с учетом контекстов - длина батча Х величина окна Х число топиков
    stacked_tensor = jax.vmap(shift_batch)(shifts).transpose(1, 0, 2)
    print(f"{stacked_tensor.shape=}")
    return stacked_tensor

# phi_it_hatch_with_context - трехмерный массив с измерениями длина батча Х величина окна Х число топиков (i Х j Х t)
# где в ячейке вероятность, которую в заданной позиции i вносит токен из позиции j в контексте Ci, для топика t
# окно контекста с учетом границ документов (зануляются токены вне границ)
phi_it_hatch_with_context = _get_context_tensor(batch=phi_it_hatch)
print(f"{phi_it_hatch_with_context.shape=}")

shifts=Array([ 0, -1, -2, -3, -4, -5, -6], dtype=int32)
padding.shape=(3, 10)
padding=Array([[-1., -1., -1., -1., -1., -1., -1., -1., -1., -1.],
       [-1., -1., -1., -1., -1., -1., -1., -1., -1., -1.],
       [-1., -1., -1., -1., -1., -1., -1., -1., -1., -1.]], dtype=float32)
padded_batch.shape=(26, 10)
padded_batch=Array([[-1.        , -1.        , -1.        , -1.        , -1.        ,
        -1.        , -1.        , -1.        , -1.        , -1.        ],
       [-1.        , -1.        , -1.        , -1.        , -1.        ,
        -1.        , -1.        , -1.        , -1.        , -1.        ],
       [-1.        , -1.        , -1.        , -1.        , -1.        ,
        -1.        , -1.        , -1.        , -1.        , -1.        ],
       [ 0.09118721,  0.11585489,  0.17614725,  0.15402782,  0.12468565,
         0.11698034,  0.04652762,  0.05900681,  0.08696249,  0.02861993],
       [ 0.1055353 ,  0.03661064,  0.05360207,  0.12943278,  0.15812047,
         0.14386164

In [46]:
batch_size = len(tokenized_data)
attn_matrix = jnp.ones(
    shape=(batch_size + ctx_len * 2, ctx_len * 2 + 1),
    dtype=bool,
)  # (len(data) + 2C, 2C + 1)
print(attn_matrix.shape) # (20 + 2 * 3, 2 * 3 )
prefix_bounds = document_bounds[: -1] + ctx_len
print(prefix_bounds)

(26, 7)
[ 3  8 12 19]


In [47]:
ignored_mask_prefix = jnp.ones((ctx_len, ctx_len), dtype=bool)  # (C, C)
ignored_mask_prefix = jnp.rot90(~jnp.triu(ignored_mask_prefix))
print(ignored_mask_prefix)
ignored_mask_prefix = jnp.tile(ignored_mask_prefix, reps=len(prefix_bounds),).T
ignored_mask_prefix

[[False False False]
 [False False  True]
 [False  True  True]]


Array([[False, False, False],
       [False, False,  True],
       [False,  True,  True],
       [False, False, False],
       [False, False,  True],
       [False,  True,  True],
       [False, False, False],
       [False, False,  True],
       [False,  True,  True],
       [False, False, False],
       [False, False,  True],
       [False,  True,  True]], dtype=bool)

In [48]:
shifts = jnp.ones((len(prefix_bounds), ctx_len), dtype=int)  # (B, C)
print(shifts)
shifts = shifts.at[:, 0].set(prefix_bounds)
print(shifts)
shifts = jnp.cumsum(shifts, axis=1)
print(shifts)
shifts = shifts.reshape(-1, 1)  # (B * C, 1)
print(shifts.T)

[[1 1 1]
 [1 1 1]
 [1 1 1]
 [1 1 1]]
[[ 3  1  1]
 [ 8  1  1]
 [12  1  1]
 [19  1  1]]
[[ 3  4  5]
 [ 8  9 10]
 [12 13 14]
 [19 20 21]]
[[ 3  4  5  8  9 10 12 13 14 19 20 21]]


In [49]:
prefix_columns = jnp.arange(ctx_len)  # (C, )
print(prefix_columns)
attn_matrix = attn_matrix.at[shifts, prefix_columns].set(ignored_mask_prefix)
print(attn_matrix.shape)

[0 1 2]
(26, 7)


In [50]:
suffix_bounds = attn_bounds[1:]  # (B, )
ignored_mask_suffix = jnp.ones((ctx_len, ctx_len), dtype=bool)  # (C, C)
print(ignored_mask_suffix)
ignored_mask_suffix = jnp.rot90(~jnp.tril(ignored_mask_suffix))  # (C, C)
# for broadcasting
print(ignored_mask_suffix)
ignored_mask_suffix = jnp.tile(
    ignored_mask_suffix,
    reps=len(suffix_bounds),
).T  # (B * C, C)
print(ignored_mask_suffix)

[[ True  True  True]
 [ True  True  True]
 [ True  True  True]]
[[ True  True False]
 [ True False False]
 [False False False]]
[[ True  True False]
 [ True False False]
 [False False False]
 [ True  True False]
 [ True False False]
 [False False False]
 [ True  True False]
 [ True False False]
 [False False False]
 [ True  True False]
 [ True False False]
 [False False False]]


In [51]:
shifts = jnp.ones((len(suffix_bounds), ctx_len), dtype=int)  # (B, C)
print(shifts)
shifts = shifts.at[:, 0].set(suffix_bounds)
print(shifts)
shifts = jnp.cumsum(shifts, axis=1)
print(shifts)
shifts = shifts.reshape(-1, 1)  # (B * C, 1)
print(shifts.T)

[[1 1 1]
 [1 1 1]
 [1 1 1]
 [1 1 1]]
[[ 5  1  1]
 [ 9  1  1]
 [16  1  1]
 [20  1  1]]
[[ 5  6  7]
 [ 9 10 11]
 [16 17 18]
 [20 21 22]]
[[ 5  6  7  9 10 11 16 17 18 20 21 22]]


In [52]:
suffix_columns = jnp.arange(ctx_len + 1, ctx_len * 2 + 1)  # (C, )
suffix_columns

Array([4, 5, 6], dtype=int32)

In [53]:
attn_matrix

Array([[ True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True,  True],
       [False,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True,  True],
       [False,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True,  True],
       [False,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [ True,

In [54]:
ignored_mask_suffix[::-1]

Array([[False, False, False],
       [ True, False, False],
       [ True,  True, False],
       [False, False, False],
       [ True, False, False],
       [ True,  True, False],
       [False, False, False],
       [ True, False, False],
       [ True,  True, False],
       [False, False, False],
       [ True, False, False],
       [ True,  True, False]], dtype=bool)

In [55]:
attn_matrix = attn_matrix.at[shifts[::-1], suffix_columns].set(ignored_mask_suffix[::-1])
attn_matrix

Array([[ True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True,  True],
       [False,  True,  True,  True,  True,  True, False],
       [ True,  True,  True,  True,  True, False, False],
       [ True,  True,  True,  True, False, False, False],
       [False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True, False],
       [False,  True,  True,  True,  True, False, False],
       [ True,  True,  True,  True, False, False, False],
       [False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True,  True],
       [False,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True, False],
       [ True,

In [56]:
# remove padding
attn_matrix = attn_matrix[ctx_len: -ctx_len]  # (I, 2C + 1)
attn_matrix

Array([[False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True,  True],
       [False,  True,  True,  True,  True,  True, False],
       [ True,  True,  True,  True,  True, False, False],
       [ True,  True,  True,  True, False, False, False],
       [False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True, False],
       [False,  True,  True,  True,  True, False, False],
       [ True,  True,  True,  True, False, False, False],
       [False, False, False,  True,  True,  True,  True],
       [False, False,  True,  True,  True,  True,  True],
       [False,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True, False],
       [ True,  True,  True,  True,  True, False, False],
       [ True,  True,  True,  True, False, False, False],
       [False, False, False,  True,  True,  True,  True],
       [False,

In [57]:
print(document_bounds)
list(enumerate(list(attn_matrix)))

[ 0  5  9 16 20]


[(0, Array([False, False, False,  True,  True,  True,  True], dtype=bool)),
 (1, Array([False, False,  True,  True,  True,  True,  True], dtype=bool)),
 (2, Array([False,  True,  True,  True,  True,  True, False], dtype=bool)),
 (3, Array([ True,  True,  True,  True,  True, False, False], dtype=bool)),
 (4, Array([ True,  True,  True,  True, False, False, False], dtype=bool)),
 (5, Array([False, False, False,  True,  True,  True,  True], dtype=bool)),
 (6, Array([False, False,  True,  True,  True,  True, False], dtype=bool)),
 (7, Array([False,  True,  True,  True,  True, False, False], dtype=bool)),
 (8, Array([ True,  True,  True,  True, False, False, False], dtype=bool)),
 (9, Array([False, False, False,  True,  True,  True,  True], dtype=bool)),
 (10, Array([False, False,  True,  True,  True,  True,  True], dtype=bool)),
 (11, Array([False,  True,  True,  True,  True,  True,  True], dtype=bool)),
 (12, Array([ True,  True,  True,  True,  True,  True,  True], dtype=bool)),
 (13, Arr

In [58]:
# ctx_len = 3, gamma = 0.6
_context_weights_1d = np.array([0.0384, 0.096 , 0.24  , 0.    , 0.24  , 0.096 , 0.0384], dtype=np.float32)
context_matrix = _context_weights_1d * attn_matrix  # (I, 2C + 1)
list(enumerate(list(context_matrix)))

[(0,
  Array([0.    , 0.    , 0.    , 0.    , 0.24  , 0.096 , 0.0384], dtype=float32)),
 (1,
  Array([0.    , 0.    , 0.24  , 0.    , 0.24  , 0.096 , 0.0384], dtype=float32)),
 (2, Array([0.   , 0.096, 0.24 , 0.   , 0.24 , 0.096, 0.   ], dtype=float32)),
 (3,
  Array([0.0384, 0.096 , 0.24  , 0.    , 0.24  , 0.    , 0.    ], dtype=float32)),
 (4,
  Array([0.0384, 0.096 , 0.24  , 0.    , 0.    , 0.    , 0.    ], dtype=float32)),
 (5,
  Array([0.    , 0.    , 0.    , 0.    , 0.24  , 0.096 , 0.0384], dtype=float32)),
 (6, Array([0.   , 0.   , 0.24 , 0.   , 0.24 , 0.096, 0.   ], dtype=float32)),
 (7, Array([0.   , 0.096, 0.24 , 0.   , 0.24 , 0.   , 0.   ], dtype=float32)),
 (8,
  Array([0.0384, 0.096 , 0.24  , 0.    , 0.    , 0.    , 0.    ], dtype=float32)),
 (9,
  Array([0.    , 0.    , 0.    , 0.    , 0.24  , 0.096 , 0.0384], dtype=float32)),
 (10,
  Array([0.    , 0.    , 0.24  , 0.    , 0.24  , 0.096 , 0.0384], dtype=float32)),
 (11,
  Array([0.    , 0.096 , 0.24  , 0.    , 0.24  , 0.0

In [71]:
# матрица весов контекста
# количество строк - позиции в батче 
# в каждой строке нормированные веса с учетом границ документов
context_matrix = _norm(context_matrix.T).T
print(f"{context_matrix=}") 

context_matrix=array([[0.        , 0.        , 0.        , 0.        , 0.64102566,
        0.25641027, 0.10256411],
       [0.        , 0.        , 0.390625  , 0.        , 0.390625  ,
        0.15625001, 0.06250001],
       [0.        , 0.14285715, 0.35714287, 0.        , 0.35714287,
        0.14285715, 0.        ],
       [0.0625    , 0.15625   , 0.39062497, 0.        , 0.39062497,
        0.        , 0.        ],
       [0.10256411, 0.25641027, 0.64102566, 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.64102566,
        0.25641027, 0.10256411],
       [0.        , 0.        , 0.4166667 , 0.        , 0.4166667 ,
        0.16666667, 0.        ],
       [0.        , 0.16666667, 0.4166667 , 0.        , 0.4166667 ,
        0.        , 0.        ],
       [0.10256411, 0.25641027, 0.64102566, 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.64102566,
        

In [62]:
print(f"{context_matrix[..., None].shape=}")
print(f"{phi_it_hatch_with_context.shape=}")

context_matrix[..., None].shape=(20, 7, 1)
phi_it_hatch_with_context.shape=(20, 7, 10)


In [84]:
# element-wise умножение матрицы весов контекста для каждой позиции (20, 7) на
# phi_it_hatch_with_context - трехмерный массив с измерениями длина батча Х величина окна Х число топиков (i Х j Х t) (20, 7, 10)
# где в ячейке вероятность, которую в заданной позиции i вносит токен из позиции j в контексте Ci, для топика t
# окно контекста с учетом границ документов (зануляются токены вне границ)
# после умножения вероятности, взвешенные по весам в контексте
theta_it = context_matrix[..., None] * phi_it_hatch_with_context
print(f"{theta_it.shape=}")

theta_it.shape=(20, 7, 10)


In [82]:
# вот как умнножаются, чтобы получилась theta_it
print(f"{context_matrix[..., None][0][4]=}")
print(f"{phi_it_hatch_with_context[0][4]=}")
print(f"{theta_it[0][4]=}")
print(f"{context_matrix[..., None][0][4] * phi_it_hatch_with_context[0][4]=}")

context_matrix[..., None][0][4]=array([0.64102566], dtype=float32)
phi_it_hatch_with_context[0][4]=Array([0.1055353 , 0.03661064, 0.05360207, 0.12943278, 0.15812047,
       0.14386164, 0.10507683, 0.11007395, 0.08883123, 0.06885507],      dtype=float32)
theta_it[0][4]=Array(0.15768561, dtype=float32)
context_matrix[..., None][0][4] * phi_it_hatch_with_context[0][4]=Array([0.06765083, 0.02346836, 0.0343603 , 0.08296973, 0.10135928,
       0.092219  , 0.06735694, 0.07056022, 0.0569431 , 0.04413787],      dtype=float32)


In [85]:
# суммирование по измерению окна контекста - остаются позиции в батче и топики
# в результате вероятность топика в данной позиции, полученная по токенам в окне контекста 
theta_it = jnp.sum(theta_it, axis=1)  # (I, T)
print(f"{theta_it.shape=}")

theta_it.shape=(20, 10)


In [90]:
phi_it = jnp.take_along_axis(
        phi,
        indices=batch[:, None],
        axis=0,
    )
print(f"{phi_it.shape=}")
theta = theta_it
print(f"{theta.shape=}")
print(f"{(phi_it * theta).shape=}")
# element-wise умножение Фи (в части токенов, относящихся к батчу) на Тету
# обе матрицы размерностью ((I, T))
# нормализация по строкам - вероятность топиков для каждой позиции в сумме единица
p_ti = _norm((phi_it * theta).T).T  # (I, T)

phi_it.shape=(20, 10)
theta.shape=(20, 10)
(phi_it * theta).shape=(20, 10)


In [94]:
# element-wise
print(f"{phi_it[0]=}")
print(f"{theta[0]=}")
print(f"{(phi_it * theta)[0]=}")

print(f"{phi_it[0][1]=}")
print(f"{theta[0][1]=}")
print(f"{(phi_it * theta)[0][1]=}")
print(f"{phi_it[0][1] * theta[0][1]=}")

phi_it[0]=Array([0.04269939, 0.05425029, 0.08248284, 0.07212517, 0.05838539,
       0.05477729, 0.02178705, 0.02763057, 0.04072112, 0.01340159],      dtype=float32)
theta[0]=Array([0.10762397, 0.04735994, 0.05892551, 0.10403094, 0.15768561,
       0.12226825, 0.11663017, 0.11516932, 0.09098475, 0.07932156],      dtype=float32)
(phi_it * theta)[0]=Array([0.00459548, 0.00256929, 0.00486034, 0.00750325, 0.00920654,
       0.00669752, 0.00254103, 0.00318219, 0.003705  , 0.00106303],      dtype=float32)
phi_it[0][1]=Array(0.05425029, dtype=float32)
theta[0][1]=Array(0.04735994, dtype=float32)
(phi_it * theta)[0][1]=Array(0.00256929, dtype=float32)
phi_it[0][1] * theta[0][1]=Array(0.00256929, dtype=float32)


In [96]:
def _calc_p_ti(
        *,
        phi: jax.Array,
        theta: jax.Array,
        batch: jax.Array
) -> tuple[jax.Array, jax.Array]:
    phi_it = jnp.take_along_axis(
        phi,
        indices=batch[:, None],
        axis=0,
    )  # (I, T)
    p_ti = _norm((phi_it * theta).T).T  # (I, T)
    return p_ti, phi_it

theta = theta_it
# phi_it - Фи (в части токенов, относящихся к батчу) вероятность темы при наличии токена w в позиции i
# theta - Тета вероятность топика в данной позиции, полученная по токенам в окне контекста позиции i
# p_ti - вероятность топика в этой позиции при наличии этого токена и окружающего его контекста

p_ti, phi_it = _calc_p_ti(
    phi=phi,
    theta=theta,
    batch=batch,
)  # (I, T)

print(f"{p_ti.shape=}")
print(f"{phi_it.shape=}")

p_ti.shape=(20, 10)
phi_it.shape=(20, 10)


In [98]:
# суммируем вероятности топика по всем позициям, получаем оценку частот топиков в батче
def _calc_n_t(*, p_ti):
    return jnp.sum(p_ti, axis=0)  # (T, )

n_t_new = _calc_n_t(p_ti=p_ti)
print(f"{n_t_new=}")

n_t_new=Array([1.7109293, 1.5929288, 1.7314843, 1.9040341, 2.2407615, 2.2293856,
       2.115903 , 1.8696146, 2.1715155, 2.4334447], dtype=float32)


In [ ]:
def _calc_phi(
        *,
        batch: jax.Array,
        phi: jax.Array,
        p_ti: jax.Array,
        grad_reg: Callable,
    ):
    # jnp.add.at - jax.numpy.ufunc.at(a, indices[, b, inplace])
    # применяет функцию ufunc к элементам a по индексам indices, b - аргумент, inplace=True имитирует по_месту 
    # здесь создает новый массив на основе нулевого массива формы phi и для токенов, полученных в батче, прибавляет p_ti
    # прибавляет столько раз, сколько встречается в батче
    phi_new = jnp.add.at(
        jnp.zeros_like(phi),
        batch,
        p_ti,
        inplace=False,
    )  # (W, T)
    phi_new -= phi * grad_reg(phi)  # (W, T)
    phi_new = _norm(phi_new)  # (W, T)
    return phi_new
